# BOQ Data Analysis

Unity Catalog: `ingestion_framework_test.bid_data_exploration`

## Setup

In [ ]:
CATALOG = "ingestion_framework_test"
SCHEMA = "bid_data_exploration"
FQ = f"{CATALOG}.{SCHEMA}"

import pandas as pd
import plotly.graph_objects as go

pd.set_option("display.max_colwidth", 120)

# Chart colors: fixed categorical order, single-hue blue ramp for magnitude.
# CATEGORY_COLORS / FLAG_COLORS are looked up by key everywhere a chart
# colors by BOQ category or flag value, so colors stay consistent across
# every chart in this notebook regardless of row order or filtering.
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQ_BLUE = {100: "#cde2fb", 250: "#86b6ef", 400: "#3987e5", 550: "#1c5cab", 700: "#0d366b"}
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRID = "#e1e0d9"
SURFACE = "#fcfcfb"

CATEGORY_ORDER = [
    "Detailed BOQ (10+ lines/vendor)",
    "Shallow (2-9 lines/vendor)",
    "Lump-sum (1 line/vendor)",
    "No priced lines at all",
]
CATEGORY_COLORS = {
    "Detailed BOQ (10+ lines/vendor)": CAT[0],
    "Shallow (2-9 lines/vendor)": CAT[1],
    "Lump-sum (1 line/vendor)": CAT[2],
    "No priced lines at all": CAT[3],
}
FLAG_COLORS = {"N": CAT[5], "Y": CAT[4], "null": INK_MUTED}


def style_fig(fig, title, height=440, showlegend=False):
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=INK_PRIMARY)),
        plot_bgcolor=SURFACE,
        paper_bgcolor=SURFACE,
        font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", color=INK_SECONDARY, size=12),
        height=height,
        margin=dict(l=70, r=30, t=60, b=60),
        showlegend=showlegend,
        legend=dict(bgcolor="rgba(0,0,0,0)", orientation="h", y=1.1, x=0),
        xaxis=dict(showgrid=False, linecolor=GRID, tickfont=dict(color=INK_MUTED)),
        yaxis=dict(showgrid=True, gridcolor=GRID, zeroline=False, tickfont=dict(color=INK_MUTED)),
    )
    return fig


def aed(x):
    if x is None or pd.isna(x):
        return "n/a"
    return f"AED {x/1e6:,.1f}M"

print("Setup complete.")

Setup complete.


## 1. Detailed vs. Lump-Sum BOQ Breakdown

### 1.1 Classification Summary

Classification by average `quotationline` rows per vendor per RFQ.

In [ ]:
boq_shape_df = spark.sql(f"""
    WITH vendor_lines AS (
        SELECT RFQNUM, VENDOR, COUNT(*) AS line_count
        FROM {FQ}.quotationline
        GROUP BY RFQNUM, VENDOR
    ),
    rfq_lines AS (
        SELECT RFQNUM,
               AVG(line_count) AS avg_lines_per_vendor,
               MAX(line_count) AS max_lines_per_vendor,
               COUNT(DISTINCT VENDOR) AS vendor_count
        FROM vendor_lines
        GROUP BY RFQNUM
    )
    SELECT r.RFQNUM, r.DESCRIPTION, r.TOTALAWVALUE, r.DETAILBOQAVAILABLE,
           rl.avg_lines_per_vendor, rl.max_lines_per_vendor, rl.vendor_count
    FROM {FQ}.rfq r
    LEFT JOIN rfq_lines rl ON r.RFQNUM = rl.RFQNUM
""").toPandas()

def categorize(avg_lines):
    if pd.isna(avg_lines):
        return "No priced lines at all"
    if avg_lines <= 1:
        return "Lump-sum (1 line/vendor)"
    if avg_lines < 10:
        return "Shallow (2-9 lines/vendor)"
    return "Detailed BOQ (10+ lines/vendor)"

boq_shape_df["category"] = boq_shape_df["avg_lines_per_vendor"].apply(categorize)

summary = (
    boq_shape_df.groupby("category")
    .agg(
        rfq_count=("RFQNUM", "count"),
        total_award_value=("TOTALAWVALUE", "sum"),
        rfqs_with_award_value=("TOTALAWVALUE", lambda s: s.notna().sum()),
    )
    .reset_index()
)
summary["pct_of_all_rfqs"] = (summary["rfq_count"] / len(boq_shape_df) * 100).round(1)
summary = summary.sort_values("rfq_count", ascending=False)

print(f"Total RFQs in scope: {len(boq_shape_df):,}")
summary

Total RFQs in scope: 30,338


,category,rfq_count,total_award_value,rfqs_with_award_value,pct_of_all_rfqs
1,Lump-sum (1 line/vendor),10903,62807632527.3623,8226,35.9
3,Shallow (2-9 lines/vendor),9893,28162918449.0553,7663,32.6
2,No priced lines at all,6253,47948074.0000,2067,20.6
0,Detailed BOQ (10+ lines/vendor),3289,47297969733.5552,2689,10.8


- `total_award_value`: sums only RFQs where `TOTALAWVALUE` is populated at the header level (most rows are `null`) — directional, not comprehensive.
- `rfqs_with_award_value`: count of RFQs in each bucket with a value to sum.

In [ ]:
plot_df = summary.set_index("category").reindex(CATEGORY_ORDER).reset_index()

fig = go.Figure(go.Bar(
    x=plot_df["category"], y=plot_df["rfq_count"],
    marker_color=[CATEGORY_COLORS[c] for c in plot_df["category"]],
    text=[f"{n:,}<br>({p:.1f}%)" for n, p in zip(plot_df["rfq_count"], plot_df["pct_of_all_rfqs"])],
    textposition="outside",
))
style_fig(fig, "How many RFQs have a real, detailed BOQ? (by line count per vendor)")
fig.update_yaxes(title_text="Number of RFQs")
fig.show()

In [ ]:
value_df = plot_df[plot_df["total_award_value"].notna()]
fig = go.Figure(go.Bar(
    x=value_df["category"], y=value_df["total_award_value"].astype(float) / 1e6,
    marker_color=[CATEGORY_COLORS[c] for c in value_df["category"]],
    text=[f"AED {float(v)/1e6:,.0f}M" for v in value_df["total_award_value"]],
    textposition="outside",
))
style_fig(fig, "Total recorded award value by BOQ category (AED millions, where TOTALAWVALUE is populated)")
fig.update_yaxes(title_text="AED millions")
fig.show()

### 1.2 `DETAILBOQAVAILABLE` Flag vs. Real Structure

In [ ]:
cross = pd.crosstab(boq_shape_df["category"], boq_shape_df["DETAILBOQAVAILABLE"].fillna("null"))
cross = cross.reindex(CATEGORY_ORDER)
cross

DETAILBOQAVAILABLE,N,Y,null
category,,,
Detailed BOQ (10+ lines/vendor),17,106,3166
Shallow (2-9 lines/vendor),31,134,9728
Lump-sum (1 line/vendor),74,111,10718
No priced lines at all,19,61,6173


In [ ]:
fig = go.Figure()
for flag_val in ["N", "Y", "null"]:
    if flag_val not in cross.columns:
        continue
    fig.add_trace(go.Bar(
        name=flag_val, x=cross.index, y=cross[flag_val],
        marker_color=FLAG_COLORS[flag_val],
    ))
fig.update_layout(barmode="stack")
style_fig(fig, "DETAILBOQAVAILABLE flag vs. real BOQ structure — the flag doesn't predict the category", showlegend=True)
fig.update_yaxes(title_text="Number of RFQs")
fig.show()

- 106 of 3,289 genuinely detailed RFQs (3.2%) are flagged `Y`.
- 61 of 6,253 RFQs with zero priced lines are flagged `Y` anyway.
- Flag does not predict real BOQ structure.

## 2. Example Tenders by Category

One complete, real example per category from Section 1.1 — sized so the whole tender can be shown, not a truncated sample.

### 2.1 Detailed BOQ Example

In [ ]:
detailed_candidates = spark.sql(f"""
    WITH vendor_lines AS (
        SELECT RFQNUM, VENDOR, COUNT(*) AS line_count
        FROM {FQ}.quotationline
        GROUP BY RFQNUM, VENDOR
    ),
    rfq_lines AS (
        SELECT RFQNUM,
               AVG(line_count) AS avg_lines_per_vendor,
               COUNT(DISTINCT VENDOR) AS vendor_count,
               SUM(line_count) AS total_lines
        FROM vendor_lines
        GROUP BY RFQNUM
    )
    SELECT r.RFQNUM, r.DESCRIPTION, r.TOTALAWVALUE, r.DISCOUNT_REVISION,
           rl.avg_lines_per_vendor, rl.vendor_count, rl.total_lines
    FROM {FQ}.rfq r
    JOIN rfq_lines rl ON r.RFQNUM = rl.RFQNUM
    WHERE rl.avg_lines_per_vendor >= 10 AND rl.total_lines BETWEEN 10 AND 80
    ORDER BY rl.total_lines DESC
    LIMIT 10
""").toPandas()
detailed_candidates

In [ ]:
EXAMPLE_RFQNUM = detailed_candidates.iloc[0]["RFQNUM"] if len(detailed_candidates) else None
print("Detailed BOQ example:", EXAMPLE_RFQNUM)

detailed_example_lines = spark.sql(f"""
    SELECT VENDOR, RFQLINENUM, BOQITEMNUM, DESCRIPTION, ORDERQTY, ORDERUNIT,
           UNITCOST, LINECOST, LINECOSTWDIS, DISCOUNT_PERCENT, LINETYPE, ISAWARDED
    FROM {FQ}.quotationline
    WHERE RFQNUM = '{EXAMPLE_RFQNUM}'
    ORDER BY VENDOR, RFQLINENUM
""").toPandas()
detailed_example_lines

- No separate CIF/Erection columns.
- `BOQITEMNUM` null for most detailed BOQs (2,480/2,480 lines null on the largest example found).
- `RFQLINENUM` is the cross-vendor line-item key — 100% match on `DESCRIPTION`/`ORDERQTY`/`ORDERUNIT` across vendors, verified on a 620-line example.

### 2.2 Shallow BOQ Example

In [ ]:
shallow_candidates = spark.sql(f"""
    WITH vendor_lines AS (
        SELECT RFQNUM, VENDOR, COUNT(*) AS line_count
        FROM {FQ}.quotationline
        GROUP BY RFQNUM, VENDOR
    ),
    rfq_lines AS (
        SELECT RFQNUM,
               AVG(line_count) AS avg_lines_per_vendor,
               COUNT(DISTINCT VENDOR) AS vendor_count,
               SUM(line_count) AS total_lines
        FROM vendor_lines
        GROUP BY RFQNUM
    )
    SELECT r.RFQNUM, r.DESCRIPTION, r.TOTALAWVALUE,
           rl.avg_lines_per_vendor, rl.vendor_count, rl.total_lines
    FROM {FQ}.rfq r
    JOIN rfq_lines rl ON r.RFQNUM = rl.RFQNUM
    WHERE rl.avg_lines_per_vendor > 1 AND rl.avg_lines_per_vendor < 10
      AND rl.total_lines BETWEEN 4 AND 60
    ORDER BY rl.total_lines DESC
    LIMIT 10
""").toPandas()
shallow_candidates

In [ ]:
SHALLOW_EXAMPLE = shallow_candidates.iloc[0]["RFQNUM"] if len(shallow_candidates) else None
print("Shallow BOQ example:", SHALLOW_EXAMPLE)

shallow_example_lines = spark.sql(f"""
    SELECT VENDOR, RFQLINENUM, BOQITEMNUM, DESCRIPTION, ORDERQTY, ORDERUNIT,
           UNITCOST, LINECOST, LINETYPE, ISAWARDED
    FROM {FQ}.quotationline
    WHERE RFQNUM = '{SHALLOW_EXAMPLE}'
    ORDER BY VENDOR, RFQLINENUM
""").toPandas()
shallow_example_lines

### 2.3 Lump-Sum Example

In [ ]:
d111808 = spark.sql(f"""
    SELECT VENDOR, RFQLINENUM, BOQITEMNUM, DESCRIPTION, ORDERQTY, ORDERUNIT,
           UNITCOST, LINECOST, LINETYPE, ISAWARDED
    FROM {FQ}.quotationline
    WHERE RFQNUM = 'D-111808'
    ORDER BY LINECOST DESC
""").toPandas()
d111808

,VENDOR,RFQLINENUM,BOQITEMNUM,DESCRIPTION,ORDERQTY,ORDERUNIT,UNITCOST,LINECOST,LINETYPE,ISAWARDED
0,001565,1.0000000000,None,"Construction contract for the Replacement of Shobaisi, Samha and DRA Primary Substation in Eastern Region",1.00,LS,234651862.0000,234651862.0000,MATERIAL,0E-10
1,99443592,1.0000000000,None,"Construction contract for the Replacement of Shobaisi, Samha and DRA Primary Substation in Eastern Region",1.00,LS,221525111.0000,221525111.0000,MATERIAL,0E-10
2,99102876,1.0000000000,None,"Construction contract for the Replacement of Shobaisi, Samha and DRA Primary Substation in Eastern Region",1.00,LS,208545844.6400,208545844.6400,MATERIAL,0E-10
3,99107175,1.0000000000,None,"Construction contract for the Replacement of Shobaisi, Samha and DRA Primary Substation in Eastern Region",1.00,LS,195458685.0000,195458685.0000,MATERIAL,0E-10
4,99473989,1.0000000000,None,"Construction contract for the Replacement of Shobaisi, Samha and DRA Primary Substation in Eastern Region",1.00,LS,179400000.0000,179400000.0000,MATERIAL,0E-10
5,99456349,1.0000000000,None,"Construction contract for the Replacement of Shobaisi, Samha and DRA Primary Substation in Eastern Region",1.00,LS,161562485.0800,161562485.0800,MATERIAL,0E-10
6,9958404,1.0000000000,None,"Construction contract for the Replacement of Shobaisi, Samha and DRA Primary Substation in Eastern Region",1.00,LS,149332885.0000,149332885.0000,MATERIAL,0E-10
7,001938,1.0000000000,None,"Construction contract for the Replacement of Shobaisi, Samha and DRA Primary Substation in Eastern Region",1.00,LS,142459514.1000,142459514.1000,MATERIAL,1.0000000000


- One row per vendor for the entire tender.
- `DESCRIPTION` repeats the whole-tender description, not a BOQ item.
- `BOQITEMNUM` null, `ORDERQTY` always `1`.

In [ ]:
second_example_candidates = spark.sql(f"""
    WITH vendor_lines AS (
        SELECT RFQNUM, VENDOR, COUNT(*) AS line_count
        FROM {FQ}.quotationline
        GROUP BY RFQNUM, VENDOR
    ),
    rfq_lines AS (
        SELECT RFQNUM, AVG(line_count) AS avg_lines_per_vendor, COUNT(DISTINCT VENDOR) AS vendor_count
        FROM vendor_lines GROUP BY RFQNUM
    )
    SELECT r.RFQNUM, r.DESCRIPTION, r.TOTALAWVALUE, rl.vendor_count
    FROM {FQ}.rfq r
    JOIN rfq_lines rl ON r.RFQNUM = rl.RFQNUM
    WHERE rl.avg_lines_per_vendor <= 1 AND rl.vendor_count >= 3 AND r.TOTALAWVALUE IS NOT NULL
    ORDER BY r.TOTALAWVALUE DESC
    LIMIT 10
""").toPandas()
second_example_candidates

,RFQNUM,DESCRIPTION,TOTALAWVALUE,vendor_count
0,G-5809,Strategic Water Storage / Recovery Project in Liwa.,1611242556.0000,8
1,D-10662,"LTRA for Improvement & Extension to Water Networks in the Eastern region, Abu Dhabi.",1265000000.0000,5
2,D-10658,frame work contracts for execuation of electricity network at Western Region,780870000.0000,4
3,D-10659,FrameWork Contract for execution of Electricity Networks in the Eastern Region,647700000.0000,4
4,A-3900.1,Expansion & Reinforcement of 33 KV Network,599463543.5500,3
5,N-4554,New 400/132/22 kV Grid station in Sadiat Island,593831596.0000,3
6,N-6462,400/132kV Gridstation at Mahawi and extension of existing 400/220kV Grid Station,582782577.0000,3
7,D-10661,"LTRA for Improvement & Extension to Water Networks in the Western Region, Abu Dhabi.",550000000.0000,4
8,N-5208,Supply and Installation of New 132/22kV Primary S.S for Raha- A Development and New 132/11kV Primary S.S for Danet D...,528218630.0000,4
9,N-4914,New 3x132/22kV Substations at Reem Island and Associated Cable Works,463200000.0000,5


In [ ]:
SECOND_EXAMPLE = second_example_candidates.iloc[0]["RFQNUM"] if len(second_example_candidates) else None

second_example_lines = spark.sql(f"""
    SELECT VENDOR, RFQLINENUM, BOQITEMNUM, DESCRIPTION, ORDERQTY, ORDERUNIT,
           UNITCOST, LINECOST, LINETYPE, ISAWARDED
    FROM {FQ}.quotationline
    WHERE RFQNUM = '{SECOND_EXAMPLE}'
    ORDER BY LINECOST DESC
""").toPandas()
print("Second lump-sum example:", SECOND_EXAMPLE)
second_example_lines

Second lump-sum example: G-5809


,VENDOR,RFQLINENUM,BOQITEMNUM,DESCRIPTION,ORDERQTY,ORDERUNIT,UNITCOST,LINECOST,LINETYPE,ISAWARDED
0,001007,1.0000000000,None,Artificial Water,1.00,LS,2447616023.8400,2447616023.8400,SERVICE,0E-10
1,001798,1.0000000000,None,Artificial Water,1.00,LS,2030339511.2400,2030339511.2400,SERVICE,0E-10
2,003211,1.0000000000,None,Artificial Water,1.00,LS,1800383109.8800,1800383109.8800,SERVICE,0E-10
3,001686,1.0000000000,None,Artificial Water,1.00,LS,1687365705.3200,1687365705.3200,SERVICE,0E-10
4,001212,1.0000000000,None,Artificial Water,1.00,LS,1677907651.6000,1677907651.6000,SERVICE,0E-10
5,001671,1.0000000000,None,Artificial Water,1.00,LS,1652884795.3500,1652884795.3500,SERVICE,0E-10
6,002599,1.0000000000,None,Artificial Water,1.00,LS,1634260040.8700,1634260040.8700,SERVICE,0E-10
7,9918338,1.0000000000,None,Artificial Water,1.00,LS,1611242556.0000,1611242556.0000,SERVICE,1.0000000000


### 2.4 No Pricing Data Example

In [ ]:
no_pricing_candidates = spark.sql(f"""
    SELECT r.RFQNUM, r.DESCRIPTION, r.STATUS, r.ORGID, r.ENTERDATE, r.TOTALAWVALUE
    FROM {FQ}.rfq r
    WHERE NOT EXISTS (
        SELECT 1 FROM {FQ}.quotationline ql WHERE ql.RFQNUM = r.RFQNUM
    )
    ORDER BY r.ENTERDATE DESC
    LIMIT 10
""").toPandas()
no_pricing_candidates

In [ ]:
NO_PRICING_EXAMPLE = no_pricing_candidates.iloc[0]["RFQNUM"] if len(no_pricing_candidates) else None
print("No pricing data example:", NO_PRICING_EXAMPLE)

no_pricing_header = spark.sql(f"""
    SELECT RFQNUM, DESCRIPTION, STATUS, ORGID, SITEID, ENTERDATE, TOTALAWVALUE, DETAILBOQAVAILABLE
    FROM {FQ}.rfq
    WHERE RFQNUM = '{NO_PRICING_EXAMPLE}'
""").toPandas()
no_pricing_header

In [ ]:
no_pricing_vendors = spark.sql(f"""
    SELECT VENDOR, CONTACT, BIDSTATUS, BIDSTATUSDATE
    FROM {FQ}.rfqvendor
    WHERE RFQNUM = '{NO_PRICING_EXAMPLE}'
    ORDER BY VENDOR
""").toPandas()
no_pricing_vendors

- No `quotationline` rows at all — shown instead: the RFQ header and its invited-vendor list.
- `BIDSTATUS` per vendor shows whether anyone was invited but never actually priced anything.

## 3. Round Tracking on Detailed BOQs

Uses the Section 2.1 example (`EXAMPLE_RFQNUM`).

### 3.1 Round-Revision Counter

Header-level counter: `rfq.DISCOUNT_REVISION` (moves in lockstep with `rfqvendor.POSTBID_DISCOUNT_COUNTER`).

In [ ]:
rev_dist = spark.sql(f"""
    SELECT DISCOUNT_REVISION, COUNT(*) AS n
    FROM {FQ}.rfq
    WHERE DISCOUNT_REVISION IS NOT NULL
    GROUP BY DISCOUNT_REVISION
    ORDER BY DISCOUNT_REVISION
""").toPandas()

fig = go.Figure(go.Bar(
    x=rev_dist["DISCOUNT_REVISION"].astype(int), y=rev_dist["n"],
    marker_color=SEQ_BLUE[400],
    text=rev_dist["n"], textposition="outside",
))
style_fig(fig, "How many RFQs reach each negotiation round? (rfq.DISCOUNT_REVISION)")
fig.update_xaxes(title_text="Round number", dtick=1)
fig.update_yaxes(title_text="Number of RFQs")
fig.show()

### 3.2 Line-Level Before/After

`LINECOST` (original) vs. `LINECOSTWDIS` (after discount). No round-by-round history — original vs. final only.

In [ ]:
discounted = spark.sql(f"""
    SELECT VENDOR, RFQLINENUM, LINECOST, LINECOSTWDIS
    FROM {FQ}.quotationline
    WHERE RFQNUM = '{EXAMPLE_RFQNUM}'
      AND LINECOST IS NOT NULL AND LINECOSTWDIS IS NOT NULL
      AND LINECOST != LINECOSTWDIS
    ORDER BY LINECOST DESC
    LIMIT 15
""").toPandas()

if len(discounted):
    labels = [f"{v} · line {int(l)}" for v, l in zip(discounted["VENDOR"], discounted["RFQLINENUM"])]
    fig = go.Figure()
    for lab, before, after in zip(labels, discounted["LINECOST"], discounted["LINECOSTWDIS"]):
        fig.add_trace(go.Scatter(
            x=[before, after], y=[lab, lab], mode="lines",
            line=dict(color=INK_MUTED, width=2), showlegend=False,
        ))
    fig.add_trace(go.Scatter(
        x=discounted["LINECOST"], y=labels, mode="markers",
        name="Original quote", marker=dict(color=SEQ_BLUE[700], size=11),
    ))
    fig.add_trace(go.Scatter(
        x=discounted["LINECOSTWDIS"], y=labels, mode="markers",
        name="After discount", marker=dict(color=SEQ_BLUE[250], size=11),
    ))
    style_fig(fig, "Original vs. discounted price, per line item (sample)", height=460, showlegend=True)
    fig.update_xaxes(title_text="AED")
    fig.show()
else:
    print("No discounted lines for this example -- see EXAMPLE_RFQNUM in Section 2.1.")

**Limitation:** no round-number or change-timestamp column exists on `quotationline` — intermediate rounds are not preserved, only original vs. final state. See Question 4b.

## 4. Open Questions for TAQA

### 4a. Vendor Names

`rfqvendor.VENDOR` is a bare code, not a company name. Search across all columns in the schema for name-like fields:

In [ ]:
name_cols = spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM {CATALOG}.information_schema.columns
    WHERE table_schema = '{SCHEMA}'
      AND (column_name ILIKE '%NAME%' OR column_name ILIKE '%VENDOR%' OR column_name ILIKE '%COMPANY%')
    ORDER BY table_name, column_name
""").toPandas()
name_cols

,table_name,column_name,data_type
0,altquotationline,MANUFACTURERNAME,STRING
1,altquotationline,VENDOR,STRING
2,altquotationline,VENDORPACKCODE,STRING
3,altquotationline,VENDORPACKQUANTITY,STRING
4,altquotationline,VENDORWAREHOUSE,STRING
5,docinfo,DMSNAME,STRING
6,docinfo,URLNAME,STRING
7,quotationline,VENDOR,STRING
8,quotationline,VENDORPACKCODE,STRING
9,quotationline,VENDORPACKQUANTITY,STRING


- No column resolves vendor code → company name.
- `CONTACT` = person's name at the vendor; `MANUFACTURERNAME` = product manufacturer, not the bidder.
- **Question:** is there a vendor/company master table we don't have access to?

### 4b. Round Revision History

- `DISCOUNT_REVISION`/`POSTBID_DISCOUNT_COUNTER` show how many rounds occurred.
- `LINECOST`/`LINECOSTWDIS` show net effect only — no per-round history.
- **Question:** does round-by-round history exist anywhere in Maximo outside these 7 tables?

### 4c. WDIS as Confirmed Price

Match test: awarded vendor's `TOTALAWARDCOSTWDIS` vs. header `TOTALAWVALUE`.

In [ ]:
wdis_check = spark.sql(f"""
    SELECT rv.RFQNUM, r.TOTALAWVALUE AS rfq_header_award_value,
           rv.VENDOR, rv.TOTALAWARDCOSTWDIS AS vendor_wdis_value,
           ABS(r.TOTALAWVALUE - rv.TOTALAWARDCOSTWDIS) < 1 AS values_match
    FROM {FQ}.rfqvendor rv
    JOIN {FQ}.rfq r ON rv.RFQNUM = r.RFQNUM
    WHERE rv.ISAWARDED = 1
      AND rv.TOTALAWARDCOSTWDIS IS NOT NULL AND rv.TOTALAWARDCOSTWDIS > 0
      AND r.TOTALAWVALUE IS NOT NULL
    ORDER BY r.TOTALAWVALUE DESC
    LIMIT 15
""").toPandas()
wdis_check

In [ ]:
if len(wdis_check):
    match_rate = wdis_check["values_match"].mean() * 100
    print(f"Match rate on this sample: {match_rate:.0f}% ({wdis_check['values_match'].sum()}/{len(wdis_check)})")

    plot_df = wdis_check.head(8)
    labels = plot_df["RFQNUM"] + " · " + plot_df["VENDOR"]
    fig = go.Figure()
    fig.add_trace(go.Bar(
        name="rfq.TOTALAWVALUE (header)", x=labels, y=plot_df["rfq_header_award_value"].astype(float) / 1e6,
        marker_color=CAT[0],
    ))
    fig.add_trace(go.Bar(
        name="rfqvendor.TOTALAWARDCOSTWDIS", x=labels, y=plot_df["vendor_wdis_value"].astype(float) / 1e6,
        marker_color=CAT[1],
    ))
    fig.update_layout(barmode="group")
    style_fig(fig, "Header award value vs. vendor WDIS value -- do they match?", height=460, showlegend=True)
    fig.update_yaxes(title_text="AED millions")
    fig.show()
else:
    print("No rows with both values populated in this sample -- widen the LIMIT above if needed.")

- Match rate near 100% would confirm `TOTALAWARDCOSTWDIS` as the final award price.
- **Question:** confirm this is the authoritative price field, and whether `TOTALAWARDCOSTWITHTAXWDIS` (incl. tax) should drive comparisons instead.

### 4d. `altquotationline`

Schema near-identical to `quotationline`, plus `ALTQUOTATIONLINEID` (PK) and `ALTQUOTLINEUID`.

In [ ]:
alt_vs_base = spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {FQ}.altquotationline) AS alt_line_count,
        (SELECT COUNT(*) FROM {FQ}.quotationline) AS base_line_count
""").toPandas()

alt_n, base_n = alt_vs_base.iloc[0]["alt_line_count"], alt_vs_base.iloc[0]["base_line_count"]
fig = go.Figure(go.Bar(
    x=["quotationline (base lines)", "altquotationline (alternates)"],
    y=[base_n, alt_n],
    marker_color=[SEQ_BLUE[550], SEQ_BLUE[250]],
    text=[f"{base_n:,}", f"{alt_n:,}"], textposition="outside",
))
style_fig(fig, f"Alternates are rare: {alt_n/base_n*100:.1f}% of base line volume")
fig.update_yaxes(type="log", title_text="Row count (log scale)")
fig.show()

- `ALTQUOTLINEUID` → `quotationline.QUOTATIONLINEID` join returns zero rows.
- `QL2` values (`QUOTED`/`TNA`/`NOQUOTE`/`Cancel`/`CNA`) suggest a technical-acceptance status — unconfirmed.
- **Question:** what is `altquotationline` used for operationally, and what does `ALTQUOTLINEUID` reference?

## 5. Summary

- 10.8% of RFQs have a detailed, line-by-line BOQ; 35.9% lump-sum; 32.6% shallow; 20.6% no priced lines.
- Lump-sum tenders hold the most recorded value of any category (AED 62.8B).
- `DETAILBOQAVAILABLE` does not predict real BOQ structure.
- Section 2 shows one complete real example per category.
- Round tracking: header-level counter + line-level before/after only, no round-by-round history.
- Four open questions for TAQA (Section 4): vendor names, round history, WDIS confirmation, `altquotationline` purpose.